# Unsupervised learning: groups without names

```{admonition} Question
:class: tip
What groups does a model find when it is given no labels?
```

**In this notebook:** 1 · k-means on two counts  2 · three groups on the map  3 · how many groups  4 · who names a group

## 1 · k-means on two counts

**Clustering** puts rows into groups so that the rows of a group are alike. The land-use label is put aside: the method sees only the counts.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

import helpers as h

hexes, X, y, W = h.load()        # the table, the features, the labels, the neighbours
SEED = 0

**k-means** needs one number from us: K, the number of groups. It starts from K hexagons picked at random as centres and repeats two steps until no hexagon changes its group.

In [ ]:
def kmeans_steps(points, k=3, steps=8, seed=3):
    """k-means written out: returns, for every step, the centres and the group of every point."""
    centres = points[np.random.default_rng(seed).choice(len(points), k, replace=False)]
    history = []
    for _ in range(steps):
        group = np.argmin(((points[:, None, :] - centres[None, :, :]) ** 2).sum(axis=2), axis=1)      # step 1: the nearest centre
        history.append((centres.copy(), group))
        centres = np.array([points[group == g].mean(axis=0) if (group == g).any() else centres[g] for g in range(k)])   # step 2: move the centres
    return history

Two of the 16 counts, `car` and `play`, because two counts can be drawn.

In [ ]:
two_counts = X[:, [h.FEATURES.index("car"), h.FEATURES.index("play")]]
steps_2d = kmeans_steps(two_counts)
print("hexagons that change their group from one step to the next:",
      [int((steps_2d[i][1] != steps_2d[i - 1][1]).sum()) for i in range(1, len(steps_2d))])
h.plot_kmeans_step(two_counts, steps_2d, len(steps_2d) - 1);

<div class="frames" data-name="kmeans_steps" data-prefix="step" data-first="1"></div>

*The stars are the centres. The colours only tell the groups apart.*

**Result.** 244 hexagons change their group at the second step, then 34, 18, 10, 5 and 2. At the eighth step none does, and k-means stops.

## 2 · Three groups on the map

Now all 16 counts, each on the same scale so that no count outweighs the others. k-means runs for K from 2 to 8. The maps show K = 3.

In [ ]:
Zk = StandardScaler().fit_transform(X)
KS = list(range(2, 9))
kmeans = {k: KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(Zk) for k in KS}
groups = kmeans[3].labels_
h.map_kmeans_and_labels(hexes, groups, y);

The groups form patches, although k-means was given no coordinates. With another K the same hexagons fall into other groups.

<div class="frames" data-name="kmeans" data-labels="K = 2|K = 3|K = 4|K = 5|K = 6|K = 7|K = 8"></div>

## 3 · How many groups?

Nothing in the data says what K should be. The **elbow method** plots, for every K, the within-cluster sum of squares: how far the hexagons lie from the centre of their group. One looks for the K after which it falls only slowly.

In [ ]:
wcss = pd.Series({k: kmeans[k].inertia_ for k in KS})
print("fall of the sum of squares with each added group:", {f"{k - 1} to {k}": f"{1 - wcss[k] / wcss[k - 1]:.0%}" for k in KS[1:]})
h.plot_elbow(wcss);

**Result.** The third group lowers the sum of squares by 22%, the fourth by 12%, the fifth by 7%. The curve bends between 3 and 4 groups. Real data rarely give a sharp elbow. We keep K = 3, the number of land-use classes.

## 4 · Who names a group?

k-means returns group 1, 2 and 3, without names. The table counts the labels in each group and the points of interest (POIs) of its typical hexagon.

In [ ]:
table = pd.crosstab(pd.Series(groups + 1, name="group"), pd.Series(y, name="land use"))
table["hexagons"] = table.sum(axis=1)
table["median POIs"] = pd.Series(hexes["poi_total"].to_numpy()).groupby(groups + 1).median().astype(int)
named = table[h.CLASSES].idxmax(axis=1)                       # the most frequent land use of each group
agree = float((named.reindex(groups + 1).to_numpy() == y).mean())
print(table.to_string())
print("\nmost frequent land use of each group:", named.to_dict())
print(f"named after it, {agree:.0%} of the hexagons get their own label (always answering green: {(y == 'green').mean():.0%})")

**Result.** No group is mostly `work`. The groups are three levels of activity: 4, 69 and 242 POIs in a typical hexagon. A clustering answers the question that the features pose, here how much there is in a hexagon.

How often does a hexagon share its group with a hexagon that touches it?

In [ ]:
same_group = np.mean([np.mean(groups[W[i].indices] == groups[i]) for i in range(len(groups)) if W[i].nnz])
group_chance = (pd.Series(groups).value_counts(normalize=True) ** 2).sum()
print(f"same group as a neighbour: {same_group:.0%}     if the groups were scattered at random: {group_chance:.0%}")

**Result.** 73% against 49%: hexagons with similar counts lie next to each other. Naming the groups needed the label, an answer from outside the counts, and then 72% of the hexagons get their own label.

**Next:** [supervised learning](supervised)